# C2.1 Production RAG Architecture

This lesson is production-first: decide when to use RAG, build a corpus, evaluate chunking and retrieval, and assemble a grounded RAG pipeline.

Learning goals:
- Choose RAG vs fine-tuning with clear tradeoffs.
- Compare chunking strategies with retrieval metrics.
- Implement query rewriting and measure its impact.
- Build a complete RAG pipeline without external downloads.

## RAG vs fine-tuning decision

**Direct answer for this program:** use RAG for knowledge-grounded applications. Fine-tuning changes behavior, not knowledge.

Why RAG dominates production for knowledge-grounded apps:
- Retrieves fresh, private, and auditable data at request time.
- Keeps model weights stable while knowledge changes frequently.
- Reduces hallucinations by grounding responses in retrieved evidence.

Why fine-tuning is often impractical for product teams without training infra:
- Requires labeled data, GPUs, and training pipelines.
- Updates are slow and expensive when knowledge changes.
- Hard to trace or cite sources for answers.

Tradeoffs (cost, maintenance, recency):

| Dimension | RAG | Fine-tuning |
| --- | --- | --- |
| Cost | Low upfront, higher per-query | High upfront, lower per-query |
| Maintenance | Update index quickly | Retrain to update knowledge |
| Recency | Immediate | Delayed until retrain |

## Chunking strategies

No single method is universally best. Choose based on data type, retrieval goals, and system architecture.

Typical fits:
- Sentence-aware (recursive/sentence-level) for general-purpose RAG when you want coherent context without heavy compute.
- Semantic chunking for topic-heavy, long documents where precision matters and extra compute is acceptable.
- Fixed-size (sliding window) for high-throughput pipelines where predictability matters most.

Detailed breakdown:

1. Sentence-aware (recursive/sentence-level)
- What it does: splits text by logical units like sentences or paragraphs.
- Best fit: standard RAG use cases that need coherent context.
- Pros: preserves complete thoughts and avoids mid-sentence breaks.
- Cons: inconsistent chunk sizes can affect models expecting uniform inputs.

2. Semantic chunking
- What it does: uses embeddings or NLP similarity to split when topics shift.
- Best fit: long research papers, dense manuals, or multi-topic documents.
- Pros: high retrieval precision because chunks stay concept-cohesive.
- Cons: more complex and compute-heavy because it requires embedding to chunk.

3. Fixed size (sliding window)
- What it does: splits at a fixed token or character count with overlap.
- Best fit: logs, structured reports, or large datasets where throughput matters.
- Pros: simple, fast, and predictable latency.
- Cons: breaks sentences and can dilute semantic meaning.

Tradeoff snapshot (no universal winner):

| Method | Coherence | Compute | Latency predictability | Typical fit |
| --- | --- | --- | --- | --- |
| Sentence-aware | High | Medium | Medium | General RAG |
| Semantic | Highest | High | Low | Topic-heavy docs |
| Fixed-size | Low to Medium | Low | High | High-throughput |

Chunk size tradeoff:
- Too small: loses context and harms retrieval quality.
- Too large: mixes topics and adds irrelevant context.
- Balanced sizes preserve meaning and keep retrieval precise.

In [39]:
from pathlib import Path
import csv
import json
import math
import re

import numpy as np

In [ ]:

DATA_DIR = Path('data') / 'C2.1 corpus'
DATA_DIR.mkdir(parents=True, exist_ok=True)

for path in DATA_DIR.iterdir():
    if path.is_file():
        path.unlink()

# ── 60-document multi-domain corpus ──────────────────────────────────────────
# Domains: Finance (15), Education (15), Healthcare (10),
#          Technology (10), AI/ML (5), Legal (5)
# Each document: realistic multi-sentence body + one evaluation query + answer
raw_docs = [
    # ── FINANCE (15) ─────────────────────────────────────────────────────────
    ('finance', 'Loan Amortization Explained',
     'Loan amortization distributes equal monthly payments across the full loan term. '
     'Early payments are weighted heavily toward interest while later payments reduce principal. '
     'A 30-year mortgage at 6 percent APR has a fixed monthly payment calculated from the amortization formula. '
     'Borrowers who make extra principal payments shorten the loan term and reduce total interest paid.',
     'How does loan amortization distribute payments?', 'equal monthly payments'),

    ('finance', 'Portfolio Diversification Theory',
     'Modern portfolio theory shows that diversification reduces unsystematic risk in a portfolio. '
     'Harry Markowitz demonstrated that combining uncorrelated assets lowers overall portfolio volatility. '
     'The efficient frontier represents portfolios with maximum return for each given risk level. '
     'Investors should hold a mix of equities, bonds, and alternatives to optimize the risk-return tradeoff.',
     'What reduces portfolio risk through asset combination?', 'diversification'),

    ('finance', 'Price-to-Earnings Ratio',
     'The price-to-earnings ratio compares a company stock price to its earnings per share. '
     'A high P/E ratio signals that investors expect strong future growth from the company. '
     'Value investors prefer stocks with low P/E ratios relative to industry peers. '
     'The P/E ratio is one of the most widely used equity valuation metrics in fundamental analysis.',
     'What does the P/E ratio compare?', 'stock price to its earnings per share'),

    ('finance', 'Central Bank Interest Rate Policy',
     'Central banks raise interest rates to control inflation by making borrowing more expensive. '
     'Lower interest rates stimulate economic growth by encouraging consumer spending and business investment. '
     'The Federal Reserve uses the federal funds rate as its primary monetary policy tool. '
     'Rate changes ripple through mortgage rates, credit card rates, and bond yields across the economy.',
     'Why do central banks raise interest rates?', 'control inflation'),

    ('finance', 'Bond Duration and Interest Rate Risk',
     'Bond duration measures the sensitivity of a bond price to changes in interest rates. '
     'Longer-duration bonds experience greater price drops when market interest rates rise. '
     'Portfolio managers use duration matching to hedge interest rate risk in fixed-income portfolios. '
     'A bond with a duration of five years loses approximately five percent of value for each one-percent rate increase.',
     'What measures a bond sensitivity to interest rate changes?', 'duration'),

    ('finance', 'Options Contracts Calls and Puts',
     'A call option gives the buyer the right to purchase an asset at the strike price before expiry. '
     'A put option gives the buyer the right to sell an asset at the strike price before expiry. '
     'Options are used for hedging existing positions or speculating on directional price movements. '
     'The Black-Scholes model is the standard framework for pricing European options on non-dividend stocks.',
     'What right does a call option give the buyer?', 'right to purchase'),

    ('finance', 'Credit Scores and Lending Decisions',
     'Credit scores quantify a borrower likelihood of defaulting on a loan repayment. '
     'FICO scores range from 300 to 850 and are calculated from payment history, debt levels, and credit age. '
     'Lenders use credit scores to set loan approval thresholds and determine the applicable interest rate. '
     'A score above 750 typically qualifies borrowers for the best available rates from major lenders.',
     'What do credit scores measure?', 'likelihood of defaulting'),

    ('finance', 'Compound Interest and Long-Term Growth',
     'Compound interest earns returns on both the principal and all previously accumulated interest. '
     'The time value of money states that a dollar today is worth more than a dollar in the future. '
     'An investment of one thousand dollars at seven percent annual compound interest grows to about 7,612 dollars in 30 years. '
     'Starting to invest early dramatically increases the compounding effect over a long horizon.',
     'Why is compound interest powerful for long-term savings?', 'earns returns on principal and accumulated interest'),

    ('finance', 'Inflation and Purchasing Power',
     'Inflation erodes the purchasing power of money held in cash over time. '
     'The Consumer Price Index measures average price changes across a basket of goods and services. '
     'Real investment returns subtract the inflation rate from nominal returns to show actual purchasing power gain. '
     'Investors use Treasury Inflation-Protected Securities to hedge portfolios against sustained inflation risk.',
     'How does inflation affect the value of money?', 'erodes purchasing power'),

    ('finance', 'Financial Statement Analysis',
     'The income statement reports a company revenue, expenses, and net income over a reporting period. '
     'The balance sheet shows assets, liabilities, and shareholders equity at a single point in time. '
     'The cash flow statement tracks cash inflows and outflows from operations, investing, and financing activities. '
     'Analysts use all three statements together to assess a company overall financial health and sustainability.',
     'What does the income statement report?', 'revenue, expenses, and net income'),

    ('finance', 'Index Funds and Passive Investing',
     'Index funds track a market benchmark such as the S&P 500 and hold all constituent securities. '
     'Passive funds have significantly lower expense ratios than actively managed funds. '
     'Decades of data show most actively managed funds underperform their benchmark over ten-year periods. '
     'Low-cost index investing is the standard recommendation for retail investors building long-term wealth.',
     'Why do index funds have lower costs than active funds?', 'lower expense ratios'),

    ('finance', 'Risk and Return in Capital Markets',
     'The risk-return tradeoff states that higher expected returns require the investor to accept higher risk. '
     'Equities historically earn higher returns than bonds but with substantially greater price volatility. '
     'Beta measures a stock sensitivity to market movements relative to a benchmark like the S&P 500. '
     'Investors position themselves on the risk-return spectrum based on their time horizon and risk tolerance.',
     'What does the risk-return tradeoff state?', 'higher returns require accepting higher risk'),

    ('finance', 'Liquidity Risk in Banking',
     'Liquidity risk is an institution inability to meet its short-term financial obligations when due. '
     'Banks maintain pools of liquid assets to fund unexpected deposit withdrawals without fire sales. '
     'The liquidity coverage ratio requires banks to hold high-quality liquid assets for a 30-day stress scenario. '
     'Bank runs occur when depositors lose confidence and simultaneously withdraw funds beyond available liquidity.',
     'What is liquidity risk in banking?', 'inability to meet short-term obligations'),

    ('finance', 'Revenue Recognition Accounting',
     'Revenue recognition determines when a company records revenue in its financial statements. '
     'Under US GAAP revenue is recognized when it is earned and collectible, not simply when cash is received. '
     'The ASC 606 standard requires revenue to be recognized as contractual performance obligations are satisfied. '
     'Misapplied revenue recognition is a frequent source of financial restatements and regulatory enforcement.',
     'When is revenue recognized under accounting standards?', 'performance obligations are satisfied'),

    ('finance', 'Hedge Fund Investment Strategies',
     'Hedge funds use strategies including long-short equity, global macro, and statistical arbitrage. '
     'Long-short equity funds buy undervalued stocks and short overvalued stocks to earn market-neutral returns. '
     'Global macro funds take positions based on broad economic trends across countries and asset classes. '
     'Hedge funds typically charge a two-percent management fee and a twenty-percent performance allocation.',
     'What is the long-short equity strategy?', 'buy undervalued and short overvalued stocks'),

    # ── EDUCATION (15) ───────────────────────────────────────────────────────
    ('education', "Bloom's Taxonomy of Learning Objectives",
     "Bloom's Taxonomy classifies learning objectives into six cognitive levels: remember, understand, apply, analyze, evaluate, and create. "
     'Higher-order thinking skills like analysis and synthesis require deeper engagement than simple recall. '
     "Teachers use Bloom's Taxonomy to design assessments that target the appropriate cognitive demand. "
     'Moving students from memorizing facts to creating original ideas is a hallmark of higher-order instruction.',
     "What are the six levels in Bloom's Taxonomy?", 'remember, understand, apply, analyze, evaluate, and create'),

    ('education', 'Formative vs Summative Assessment',
     'Formative assessments monitor student learning continuously during instruction to inform teaching decisions. '
     'Summative assessments evaluate cumulative student achievement at the end of a unit, semester, or course. '
     'Quizzes, exit tickets, and peer reviews are widely used formative assessment tools in classrooms. '
     'Final exams and standardized tests are summative assessments used to assign grades or certifications.',
     'What is the purpose of formative assessment?', 'monitor student learning during instruction'),

    ('education', 'Zone of Proximal Development',
     "Vygotsky's Zone of Proximal Development describes the gap between what a learner can do alone and with expert guidance. "
     'Effective instruction targets this zone to stretch learners just beyond their current independent ability. '
     'Scaffolding provides temporary structured support that is gradually removed as learner competence grows. '
     'Peer collaboration and targeted teacher feedback are practical tools for supporting learning within this zone.',
     'What does the Zone of Proximal Development describe?', 'gap between what a learner can do alone and with guidance'),

    ('education', 'Active Learning Strategies in the Classroom',
     'Active learning engages students directly in the learning process through structured activities requiring thinking. '
     'Think-pair-share, case studies, and problem-based learning are established active learning approaches. '
     'Research consistently shows active learning produces better retention and transfer than passive lecture. '
     'Flipping the classroom moves direct instruction to pre-class videos, freeing class time for active application.',
     'How does active learning improve student retention?', 'engages students through activities requiring thinking'),

    ('education', 'Universal Design for Learning',
     'Universal Design for Learning provides multiple means of representation, engagement, and expression for all students. '
     'UDL removes learning barriers by building flexibility into curriculum materials and instructional methods. '
     'Offering content in text, audio, and video formats addresses diverse learner preferences and accessibility needs. '
     'UDL shifts responsibility for accessibility from individual accommodations to universal curriculum design.',
     'What three means does Universal Design for Learning provide?', 'representation, engagement, and expression'),

    ('education', 'Metacognition and Self-Regulated Learning',
     'Metacognition is the ability to monitor, evaluate, and regulate one own thinking and learning processes. '
     'Self-regulated learners set goals, select strategies, monitor progress, and reflect on their outcomes. '
     'Teaching metacognitive strategies improves academic performance across subjects and grade levels. '
     'Reflective journaling and structured think-alouds are practical classroom tools for building metacognitive awareness.',
     'What is metacognition in education?', 'monitor and regulate one own thinking'),

    ('education', 'Project-Based Learning',
     'Project-based learning engages students in sustained inquiry around complex, authentic, real-world problems. '
     'Students produce public products or presentations that demonstrate mastery rather than sitting for tests. '
     'PBL develops collaboration, communication, and critical thinking alongside disciplinary content knowledge. '
     'Teachers act as coaches who guide student inquiry rather than delivering information through direct instruction.',
     'What distinguishes project-based learning from traditional instruction?', 'sustained inquiry around real-world problems'),

    ('education', 'Differentiated Instruction',
     'Differentiated instruction tailors teaching to meet the diverse learning needs and readiness levels of students. '
     'Teachers adjust content, process, product, and environment based on ongoing assessment of student profiles. '
     'Tiered assignments allow students at different readiness levels to work toward the same core concept. '
     'Regular formative assessment informs which students need acceleration, enrichment, or additional support.',
     'What does differentiated instruction adjust for each student?', 'content, process, product, and environment'),

    ('education', 'STEM Education and Interdisciplinary Learning',
     'STEM education integrates science, technology, engineering, and mathematics through interdisciplinary problem-solving. '
     'Design challenges like robotics competitions develop technical skills alongside teamwork and communication. '
     'Early exposure to STEM activities increases long-term interest in engineering and computing careers. '
     'Effective STEM lessons connect abstract mathematical concepts to tangible engineering and design problems.',
     'What four subjects does STEM education integrate?', 'science, technology, engineering, and mathematics'),

    ('education', 'Spaced Repetition and Memory Retention',
     'Spaced repetition revisits information at systematically increasing intervals to strengthen long-term memory. '
     'The forgetting curve shows that memory for new material decays rapidly without scheduled review. '
     'Flashcard applications like Anki use spaced repetition algorithms to schedule optimal review sessions automatically. '
     'Combining spaced retrieval practice with interleaving produces the largest documented gains in retention.',
     'How does spaced repetition strengthen memory?', 'revisiting at increasing intervals'),

    ('education', 'Learning Management Systems in Education',
     'Learning Management Systems centralize course materials, assignments, grades, and communication in one platform. '
     'Moodle, Canvas, and Blackboard are widely deployed LMS platforms across K-12 and higher education globally. '
     'Learning analytics derived from LMS data help instructors identify at-risk students before they fail. '
     'Effective LMS adoption requires sustained teacher professional development and clear pedagogical alignment.',
     'What do Learning Management Systems centralize?', 'course materials, assignments, and grades'),

    ('education', 'Rubrics for Transparent Assessment',
     'Rubrics define explicit performance criteria and quality levels used to evaluate student work consistently. '
     'Analytic rubrics score each dimension separately while holistic rubrics assign a single integrated score. '
     'Sharing rubrics with students before an assignment clarifies expectations and measurably improves quality. '
     'Research shows rubrics increase inter-rater reliability and reduce subjective bias in grading.',
     'What do rubrics define to guide assessment?', 'explicit criteria and performance levels'),

    ('education', 'Inquiry-Based Learning',
     'Inquiry-based learning begins with compelling questions or problems and asks students to investigate answers. '
     'Students develop scientific reasoning by forming hypotheses, collecting data, and drawing evidence-based conclusions. '
     'The teacher role shifts from information deliverer to questioning facilitator who scaffolds the inquiry process. '
     'Inquiry learning is most effective when students possess sufficient background knowledge to engage the problem.',
     'How does inquiry-based learning start?', 'starts with questions and students investigate'),

    ('education', 'Effective Feedback Practices',
     'Effective feedback is timely, specific, and actionable, and focuses on the task rather than the learner. '
     "John Hattie's meta-analysis identified feedback as one of the most powerful influences on student achievement. "
     'Delayed or vague feedback has negligible effect on improving subsequent student performance. '
     'Structured peer feedback, when properly scaffolded, can be as effective as teacher feedback for skill development.',
     'What qualities make feedback effective in education?', 'timely, specific, and actionable'),

    ('education', 'Curriculum Alignment and Backward Design',
     'Curriculum alignment ensures learning objectives, instructional activities, and assessments form a coherent system. '
     'Backward design begins with desired outcomes and plans instruction and assessments to achieve those outcomes. '
     'Misaligned curricula produce students who study material they are never assessed on or vice versa. '
     'Regular curriculum review keeps learning objectives relevant to evolving workforce and societal requirements.',
     'What does curriculum alignment ensure?', 'objectives, instruction, and assessments are coherently linked'),

    # ── HEALTHCARE (10) ──────────────────────────────────────────────────────
    ('healthcare', 'Randomized Controlled Trials',
     'Randomized controlled trials are the gold standard for evaluating new medical treatments. '
     'Participants are randomly assigned to treatment or placebo groups to minimize selection and confounding bias. '
     'Blinding prevents participants and researchers from knowing group assignment, reducing measurement bias. '
     'Systematic reviews synthesize evidence across multiple trials to produce the strongest clinical guidance.',
     'What is the gold standard for evaluating medical treatments?', 'randomized controlled trials'),

    ('healthcare', 'Electronic Health Records',
     'Electronic Health Records store patient medical histories, diagnoses, medications, and treatment plans digitally. '
     'EHR systems improve care coordination by making records accessible across providers and care settings. '
     'Interoperability standards such as HL7 FHIR allow different EHR systems to exchange patient data securely. '
     'Poorly designed EHR interfaces increase clinician cognitive burden and contribute to professional burnout.',
     'What do Electronic Health Records store for patients?', 'medical histories, diagnoses, and treatment plans'),

    ('healthcare', 'Medication Adherence in Chronic Disease',
     'Medication adherence means patients take prescribed drugs in the correct dose at the correct time. '
     'Non-adherence accounts for approximately half of all treatment failures in patients with chronic conditions. '
     'Simplified dosing regimens, patient education, and reminder applications reliably improve adherence rates. '
     'Poor adherence costs the US healthcare system an estimated 300 billion dollars annually in avoidable costs.',
     'What is medication adherence?', 'taking prescribed drugs as directed'),

    ('healthcare', 'Preventive Care and Early Screening',
     'Preventive care reduces disease incidence and severity through early detection and targeted intervention. '
     'Screenings for cancer, diabetes, and cardiovascular disease identify conditions before symptoms develop. '
     'Vaccination prevents infectious diseases and builds community immunity through population-level herd protection. '
     'Investment in preventive care consistently delivers a higher return than treating advanced disease.',
     'How does preventive care reduce disease burden?', 'early detection and intervention'),

    ('healthcare', 'Telehealth and Remote Patient Monitoring',
     'Telehealth delivers healthcare services through video consultations, phone calls, and digital health platforms. '
     'Remote monitoring devices track vital signs such as blood pressure and blood glucose levels at home. '
     'Telehealth adoption accelerated during the COVID-19 pandemic to maintain continuity of care safely. '
     'Clinical evidence shows telehealth achieves equivalent outcomes to in-person care for many primary care conditions.',
     'What does telehealth deliver?', 'healthcare services through video and digital platforms'),

    ('healthcare', 'Patient Safety and Error Prevention',
     'Medical errors remain a leading cause of preventable patient harm and death in hospitals worldwide. '
     'Surgical checklists, standardized protocols, and simulation training measurably reduce clinical error rates. '
     'Root cause analysis investigates systemic and process failures rather than assigning individual blame. '
     'A just culture encourages staff to report near-misses and adverse events without fear of punitive response.',
     'How do hospitals reduce preventable medical errors?', 'checklists and standardized protocols'),

    ('healthcare', 'Cognitive Behavioral Therapy',
     'Cognitive Behavioral Therapy treats mental health disorders by identifying and changing negative thought patterns. '
     'CBT is an evidence-based treatment for depression, anxiety disorders, and post-traumatic stress disorder. '
     'Patients learn to recognize cognitive distortions and replace them with more accurate, balanced perspectives. '
     'CBT is typically delivered in 12 to 20 structured sessions with explicit goal-setting and outcome tracking.',
     'What does Cognitive Behavioral Therapy change?', 'negative thought patterns'),

    ('healthcare', 'Chronic Disease Management',
     'Chronic diseases such as diabetes, hypertension, and heart failure require coordinated long-term management. '
     'Self-management education empowers patients to actively monitor symptoms and adjust their own behavior. '
     'Multidisciplinary care teams coordinate treatment across physicians, nurses, dietitians, and pharmacists. '
     'Patient registries and population health analytics identify high-risk individuals who need proactive outreach.',
     'What approach is most effective for managing chronic diseases?', 'multidisciplinary care teams'),

    ('healthcare', 'Medical Imaging and Diagnostics',
     'Medical imaging modalities include X-ray, computed tomography, magnetic resonance imaging, and ultrasound. '
     'AI algorithms trained on large imaging datasets can detect tumors and anomalies with radiologist-level accuracy. '
     'MRI uses magnetic fields and radio waves and does not expose patients to ionizing radiation unlike CT and X-ray. '
     'Early imaging diagnosis enables treatment before conditions become life-threatening or surgically complex.',
     'What imaging technique avoids ionizing radiation?', 'MRI'),

    ('healthcare', 'Antibiotic Resistance',
     'Antibiotics treat bacterial infections by killing bacteria or inhibiting their reproduction. '
     'Antibiotic resistance develops when bacteria evolve mechanisms that allow them to survive antibiotic exposure. '
     'Overuse, inappropriate prescribing, and agricultural antibiotic use are the primary drivers of resistance. '
     'The World Health Organization classifies antibiotic resistance as one of the greatest global public health threats.',
     'What causes antibiotic resistance to develop?', 'overuse and inappropriate prescribing'),

    # ── TECHNOLOGY (10) ──────────────────────────────────────────────────────
    ('technology', 'Cloud Computing Service Models',
     'Cloud computing delivers on-demand computing resources over the internet using a pay-as-you-go pricing model. '
     'Infrastructure as a Service provides virtualized compute, storage, and networking managed by the cloud provider. '
     'Platform as a Service offers a managed runtime and development tools for building and deploying applications. '
     'Software as a Service delivers fully managed applications such as email, CRM, and collaboration tools over the web.',
     'What are the three cloud service delivery models?', 'IaaS, PaaS, and SaaS'),

    ('technology', 'REST API Design Principles',
     'RESTful APIs use standard HTTP methods including GET, POST, PUT, and DELETE to operate on named resources. '
     'Resources are identified by URIs and represented in standard formats such as JSON or XML. '
     'Statelessness requires each API request to carry all information needed without relying on server-side session state. '
     'API versioning with path prefixes such as slash v1 allows backward-compatible evolution of the interface.',
     'What HTTP methods do REST APIs use?', 'GET, POST, PUT, and DELETE'),

    ('technology', 'CI/CD and DevOps Pipelines',
     'DevOps integrates software development and IT operations practices to shorten the delivery cycle. '
     'Continuous Integration automatically builds and tests code changes when developers push to a shared repository. '
     'Continuous Delivery extends CI by automatically deploying tested code to staging or production environments. '
     'Automated pipelines reduce human error and provide rapid feedback between code commit and running software.',
     'What does Continuous Integration automatically do?', 'builds and tests code changes'),

    ('technology', 'Cybersecurity and Zero Trust',
     'Zero trust security assumes no user, device, or network segment is trusted by default. '
     'All access requests are verified continuously using identity, device health, and contextual signals. '
     'Least privilege principles restrict access rights to the minimum needed to perform a given role. '
     'The STRIDE framework categorizes threats as spoofing, tampering, repudiation, information disclosure, denial of service, and elevation of privilege.',
     'What does zero trust security assume?', 'no user or device is trusted by default'),

    ('technology', 'Software Testing Levels',
     'Unit tests verify the behavior of individual functions or components in complete isolation from dependencies. '
     'Integration tests check that multiple components interact correctly when wired together. '
     'End-to-end tests simulate complete user workflows through the full application stack. '
     'Test-driven development writes failing tests before code, using test failures to guide implementation decisions.',
     'What do unit tests verify?', 'individual functions in isolation'),

    ('technology', 'Containerization with Docker',
     'Containers package an application and all its runtime dependencies into a portable, isolated execution unit. '
     'Docker is the dominant container platform for building, distributing, and running containerized workloads. '
     'Kubernetes orchestrates containers at scale across clusters, handling scheduling, scaling, and self-healing. '
     'Containers enforce consistency between development, testing, and production deployment environments.',
     'What do containers package for portability?', 'application and its dependencies'),

    ('technology', 'Database Indexing',
     'Database indexes speed up query execution by enabling the engine to locate rows without a full table scan. '
     'B-tree indexes are the default index structure and efficiently support both equality and range queries. '
     'Adding too many indexes degrades write performance because every index must be updated on each insert or update. '
     'Composite indexes covering multiple columns reduce the number of index reads for multi-column filter queries.',
     'How do database indexes improve query speed?', 'locate rows without scanning the full table'),

    ('technology', 'Microservices Architecture',
     'Microservices decompose a monolithic application into small, independently deployable services. '
     'Each service owns its own data store and communicates with peers through APIs or asynchronous message queues. '
     'Independent deployability allows teams to release individual services without coordinating full-system releases. '
     'Distributed systems introduce operational complexity including latency, partial failures, and distributed tracing needs.',
     'How do microservices improve deployment flexibility?', 'independently deployable services'),

    ('technology', 'Encryption and Data Protection',
     'Encryption converts plaintext data into ciphertext that is unreadable without the correct decryption key. '
     'AES-256 is the industry standard symmetric encryption algorithm for protecting data at rest. '
     'TLS encrypts data in transit between clients and servers to prevent interception and tampering. '
     'Public-key infrastructure enables secure key exchange over untrusted networks without sharing private keys.',
     'What does encryption protect data from?', 'interception and unauthorized access'),

    ('technology', 'Version Control with Git',
     'Git is a distributed version control system that tracks every change made to source code over time. '
     'Feature branches allow developers to work in isolation without disrupting the main production codebase. '
     'Pull requests enable structured code review and discussion before merging changes into the main branch. '
     "Git's commit history forms a permanent audit trail showing what changed, when it changed, and the reason why.",
     'What does Git track for source code projects?', 'every change to source code over time'),

    # ── AI / ML (5) ──────────────────────────────────────────────────────────
    ('ai', 'Backpropagation and Gradient Descent',
     'Backpropagation computes gradients through a neural network by propagating prediction error backward. '
     'Gradient descent updates model weights iteratively in the direction that minimizes the loss function. '
     'Learning rate controls step size; too large causes training divergence, too small causes slow convergence. '
     'The Adam optimizer adapts per-parameter learning rates using first and second gradient moment estimates.',
     'What computes gradients for neural network training?', 'backpropagation'),

    ('ai', 'Transformer Architecture and Self-Attention',
     'The Transformer architecture replaces recurrence with self-attention mechanisms to model token relationships. '
     'Attention weights determine how much each token influences the contextual representation of every other token. '
     'BERT and GPT are large pre-trained Transformer models fine-tuned on downstream natural language tasks. '
     'Scaling Transformer model size and training data increases capability but requires exponentially more compute.',
     'What mechanism does the Transformer architecture use?', 'self-attention'),

    ('ai', 'Overfitting and Regularization Techniques',
     'Overfitting occurs when a model memorizes training data noise and fails to generalize to unseen examples. '
     'Dropout randomly deactivates neurons during training to prevent co-adaptation of feature detectors. '
     'L2 weight decay penalizes large parameter values by adding their squared magnitude to the loss function. '
     'Early stopping halts training when validation loss stops improving, preserving the best generalization checkpoint.',
     'What technique prevents overfitting by deactivating neurons?', 'dropout'),

    ('ai', 'RAG vs Fine-Tuning for Production LLMs',
     'Retrieval-Augmented Generation grounds LLM responses in retrieved documents to reduce hallucination. '
     'Fine-tuning adjusts model weights on labeled task-specific data to modify behavior, tone, or reasoning style. '
     'RAG is preferred for knowledge-intensive tasks because the index can be updated without any model retraining. '
     'Fine-tuning is preferred for consistent output format and specialized domain terminology not present in pretraining.',
     'When should you prefer RAG over fine-tuning?', 'dynamic knowledge bases'),

    ('ai', 'Embeddings and Vector Search',
     'Embeddings are dense fixed-length vectors that encode the semantic meaning of text in continuous space. '
     'Semantically similar texts produce embedding vectors with high cosine similarity. '
     'Vector databases such as Pinecone and Weaviate store embeddings and support approximate nearest neighbor search. '
     'Approximate nearest neighbor algorithms enable millisecond-latency retrieval across millions of vectors.',
     'What do text embeddings encode?', 'semantic meaning of text'),

    # ── LEGAL (5) ────────────────────────────────────────────────────────────
    ('legal', 'Contract Formation Requirements',
     'A valid contract requires offer, acceptance, consideration, and mutual assent between competent parties. '
     'Consideration is something of value exchanged by each party that makes the agreement legally enforceable. '
     'Contracts can be voided if formed through duress, fraud, undue influence, or material misrepresentation. '
     'Written contracts are preferred over oral agreements because they provide clear evidentiary records.',
     'What four elements are required to form a valid contract?', 'offer, acceptance, consideration, and mutual assent'),

    ('legal', 'Intellectual Property Law Overview',
     'Intellectual property law protects creations of the mind including inventions, brands, and creative works. '
     'Patents grant inventors exclusive rights to their inventions for up to 20 years from the filing date. '
     'Trademarks protect distinctive brand identifiers such as logos, names, and slogans used in commerce. '
     'Copyright arises automatically upon creation of an original work and protects expression, not ideas.',
     'What does copyright protect?', 'original creative works'),

    ('legal', 'GDPR Data Privacy Requirements',
     'The General Data Protection Regulation requires organizations to obtain explicit consent before collecting personal data. '
     'Data subjects have legal rights to access, correct, and request deletion of their personal data. '
     'Organizations experiencing a data breach must notify supervisory authorities within 72 hours of discovery. '
     'Non-compliance penalties can reach four percent of annual global turnover or 20 million euros, whichever is higher.',
     'What rights do individuals have under GDPR?', 'access, correct, and delete their data'),

    ('legal', 'Employment Law Fundamentals',
     'Employment law governs the legal relationship between employers and their employees. '
     'At-will employment allows either party to end the employment relationship at any time without cause. '
     'Anti-discrimination statutes prohibit adverse employment actions based on protected characteristics. '
     'Minimum wage laws establish a compensation floor that employers must honor regardless of any agreement.',
     'What does employment law govern?', 'relationship between employers and employees'),

    ('legal', 'Corporate Compliance Programs',
     'Compliance programs help organizations identify, prevent, and correct violations of laws and regulations. '
     'Effective programs include a written code of conduct, risk assessments, employee training, and monitoring. '
     'Leadership commitment and anonymous reporting channels are essential for building a culture of compliance. '
     'External regulatory audits assess whether compliance controls are adequately designed and actually operating.',
     'What do corporate compliance programs help organizations prevent?', 'violations of laws and regulations'),
]

documents = []
for i, (topic, title, body, query, answer) in enumerate(raw_docs, start=1):
    documents.append({
        'id': i, 'topic': topic, 'title': title,
        'body': body, 'query': query, 'answer': answer
    })

# Assign formats cyclically: first 15 → txt, next 15 → md, next 15 → jsonl, last 15 → csv
formats = ['txt'] * 15 + ['md'] * 15 + ['jsonl'] * 15 + ['csv'] * 15
for doc, fmt in zip(documents, formats):
    doc['format'] = fmt

# Write individual txt/md files
for doc in documents:
    if doc['format'] == 'txt':
        path = DATA_DIR / f"doc_{doc['id']:03d}.txt"
        path.write_text(f"Title: {doc['title']}\n{doc['body']}\n", encoding='utf-8')
    elif doc['format'] == 'md':
        path = DATA_DIR / f"doc_{doc['id']:03d}.md"
        path.write_text(f"# {doc['title']}\n\n{doc['body']}\n", encoding='utf-8')

# Write bulk jsonl
with (DATA_DIR / 'docs.jsonl').open('w', encoding='utf-8') as f:
    for doc in documents:
        if doc['format'] == 'jsonl':
            f.write(json.dumps({'id': doc['id'], 'title': doc['title'], 'body': doc['body']}) + '\n')

# Write bulk csv
with (DATA_DIR / 'docs.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'title', 'body'])
    writer.writeheader()
    for doc in documents:
        if doc['format'] == 'csv':
            writer.writerow({'id': doc['id'], 'title': doc['title'], 'body': doc['body']})

corpus_documents = documents

# Evaluation query set: 3 per domain for representative coverage
# finance(3), education(3), healthcare(3), technology(3), ai(2), legal(2) = 16 queries
EVAL_DOC_IDS = {4, 7, 9, 16, 17, 20, 31, 33, 36, 41, 43, 48, 51, 53, 56, 58}
corpus_queries = [
    {'query': doc['query'], 'doc_id': doc['id'], 'answer': doc['answer']}
    for doc in documents if doc['id'] in EVAL_DOC_IDS
]

fmt_counts = {}
for doc in documents:
    fmt_counts[doc['format']] = fmt_counts.get(doc['format'], 0) + 1

topic_counts = {}
for doc in documents:
    topic_counts[doc['topic']] = topic_counts.get(doc['topic'], 0) + 1

print('Corpus created.')
print(f'Total documents : {len(documents)}')
print(f'By format       : {fmt_counts}')
print(f'By domain       : {topic_counts}')
print(f'Eval query set  : {len(corpus_queries)} queries across {len(EVAL_DOC_IDS)} documents')
print(f'Sample (finance): "{documents[3]["title"]}" — Q: {documents[3]["query"]}')

60 domain-rich documents across Finance (15), Education (15), Healthcare (10), Technology (10), AI/ML (5), and Legal (5). Each document is 3–4 realistic sentences drawn from practitioner knowledge in that field. The 16-query evaluation set spans all six domains so learners from any background can observe retrieval working on content they recognize.

In [41]:
def load_txt(path):
    text = path.read_text(encoding='utf-8')
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    title = lines[0].replace('Title: ', '') if lines else 'Untitled'
    body = ' '.join(lines[1:]) if len(lines) > 1 else ''
    return title, body

def load_md(path):
    text = path.read_text(encoding='utf-8')
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    title = lines[0].replace('# ', '') if lines else 'Untitled'
    body = ' '.join(lines[1:]) if len(lines) > 1 else ''
    return title, body

def load_jsonl(path):
    docs = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            docs.append({
                'id': int(item['id']),
                'title': item['title'],
                'body': item['body'],
                'source': 'jsonl'
            })
    return docs

def load_csv(path):
    docs = []
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            docs.append({
                'id': int(row['id']),
                'title': row['title'],
                'body': row['body'],
                'source': 'csv'
            })
    return docs

def load_corpus(data_dir):
    docs = []
    for path in data_dir.glob('*.txt'):
        title, body = load_txt(path)
        docs.append({
            'id': int(path.stem.split('_')[1]),
            'title': title,
            'body': body,
            'source': 'txt'
        })
    for path in data_dir.glob('*.md'):
        title, body = load_md(path)
        docs.append({
            'id': int(path.stem.split('_')[1]),
            'title': title,
            'body': body,
            'source': 'md'
        })
    for path in data_dir.glob('*.jsonl'):
        docs.extend(load_jsonl(path))
    for path in data_dir.glob('*.csv'):
        docs.extend(load_csv(path))
    docs.sort(key=lambda d: d['id'])
    return docs

In [42]:
ingested_docs = load_corpus(DATA_DIR)

counts = {}
for doc in ingested_docs:
    counts[doc['source']] = counts.get(doc['source'], 0) + 1

print(f'Ingested documents: {len(ingested_docs)}')
print(f'By format: {counts}')
print(f"Example title: {ingested_docs[0]['title']}")

Ingested documents: 60
By format: {'txt': 15, 'md': 15, 'jsonl': 15, 'csv': 15}
Example title: Backpropagation and gradients


Ingestion loaded all 60 documents across four formats, and the sample title shows the parser is reading each file type correctly.

## Lab 1: Chunking baseline vs improved

Goal: use fixed-size chunking as a baseline and measure how sentence-aware and semantic chunking change retrieval quality on the same corpus. This is not about a single winner, but about tradeoffs and improvements vs baseline.

In [43]:
def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())


class SimpleTfidfVectorizer:
    def __init__(self):
        self.vocab = {}
        self.idf = None

    def fit(self, texts):
        doc_count = len(texts)
        df = {}
        for text in texts:
            terms = set(tokenize(text))
            for term in terms:
                df[term] = df.get(term, 0) + 1
        self.vocab = {term: idx for idx, term in enumerate(sorted(df.keys()))}
        self.idf = np.zeros(len(self.vocab), dtype=np.float32)
        for term, idx in self.vocab.items():
            self.idf[idx] = math.log((1 + doc_count) / (1 + df[term])) + 1.0
        return self

    def transform(self, texts):
        vectors = np.zeros((len(texts), len(self.vocab)), dtype=np.float32)
        if not self.vocab:
            return vectors
        for i, text in enumerate(texts):
            terms = tokenize(text)
            if not terms:
                continue
            counts = {}
            for term in terms:
                counts[term] = counts.get(term, 0) + 1
            for term, count in counts.items():
                idx = self.vocab.get(term)
                if idx is None:
                    continue
                tf = count / len(terms)
                vectors[i, idx] = tf * self.idf[idx]
        return vectors

    def fit_transform(self, texts):
        self.fit(texts)
        return self.transform(texts)


def split_sentences(text):
    text = text.strip()
    if not text:
        return []
    return re.split(r'(?<=[.!?])\s+', text)


def fixed_size_chunking(text, chunk_size=40, overlap=5):
    words = text.split()
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size]
        if chunk:
            chunks.append(' '.join(chunk))
    return chunks


def sentence_aware_chunking(text, max_words=60, overlap_sentences=1):
    sentences = split_sentences(text)
    chunks = []
    current_chunk = []
    current_len = 0
    for sentence in sentences:
        sentence_len = len(sentence.split())
        if current_len + sentence_len > max_words and current_chunk:
            chunks.append(' '.join(current_chunk))
            overlap = current_chunk[-overlap_sentences:]
            current_chunk = overlap.copy()
            current_len = sum(len(s.split()) for s in current_chunk)
        current_chunk.append(sentence)
        current_len += sentence_len
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks


def cosine_similarity(vec_a, vec_b):
    denom = (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)) + 1e-10
    return float(np.dot(vec_a, vec_b) / denom)


def semantic_chunking(text, similarity_threshold=0.25, min_sentences=2, max_words=80):
    sentences = split_sentences(text)
    if not sentences:
        return []
    vectorizer = SimpleTfidfVectorizer()
    vectors = vectorizer.fit_transform(sentences)
    chunks = []
    current_chunk = [sentences[0]]
    current_len = len(sentences[0].split())
    for i in range(1, len(sentences)):
        similarity = cosine_similarity(vectors[i - 1], vectors[i])
        sentence_len = len(sentences[i].split())
        should_append = (
            similarity >= similarity_threshold
            or len(current_chunk) < min_sentences
        )
        if should_append and current_len + sentence_len <= max_words:
            current_chunk.append(sentences[i])
            current_len += sentence_len
        else:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]
            current_len = sentence_len
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks

In [44]:
def build_chunks(docs, chunk_fn):
    chunks = []
    for doc in docs:
        text = f"{doc['title']}. {doc['body']}"
        for chunk in chunk_fn(text):
            chunks.append({
                'doc_id': doc['id'],
                'title': doc['title'],
                'text': chunk
            })
    return chunks


def cosine_similarity_scores(query_vec, doc_vecs):
    query_norm = np.linalg.norm(query_vec) + 1e-10
    doc_norms = np.linalg.norm(doc_vecs, axis=1) + 1e-10
    return (doc_vecs @ query_vec) / (doc_norms * query_norm)


def retrieve_top_k(query, chunks, vectorizer, chunk_vectors, top_k=3):
    query_vec = vectorizer.transform([query])[0]
    scores = cosine_similarity_scores(query_vec, chunk_vectors)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in top_indices]


def evaluate_chunking(method_name, chunk_fn, docs, queries, top_k=3):
    chunks = build_chunks(docs, chunk_fn)
    avg_len = float(np.mean([len(c['text'].split()) for c in chunks]))
    vectorizer = SimpleTfidfVectorizer()
    chunk_vectors = vectorizer.fit_transform([c['text'] for c in chunks])
    top1 = 0
    top3 = 0
    mrr_scores = []
    for item in queries:
        retrieved = retrieve_top_k(item['query'], chunks, vectorizer, chunk_vectors, top_k=top_k)
        answer = item['answer'].lower()
        found_rank = None
        for idx, chunk in enumerate(retrieved):
            if answer in chunk['text'].lower():
                found_rank = idx + 1
                break
        if found_rank == 1:
            top1 += 1
        if found_rank is not None and found_rank <= top_k:
            top3 += 1
        mrr_scores.append(1.0 / found_rank if found_rank else 0.0)
    total = len(queries)
    return {
        'method': method_name,
        'chunks': len(chunks),
        'avg_len': avg_len,
        'top1': top1 / total if total else 0.0,
        'top3': top3 / total if total else 0.0,
        'mrr': float(np.mean(mrr_scores)) if mrr_scores else 0.0
    }

In [ ]:

# ── Comparison corpus: one multi-topic doc per domain pair ────────────────────
# Documents are intentionally long to create real chunking tradeoffs.
comparison_docs = [
    {
        'id': 301,
        'title': 'Central Bank Policy and Bond Duration',
        'body': (
            'Central banks raise interest rates to control inflation by making borrowing more expensive. '
            'When rates rise, existing bond prices fall because newer bonds offer higher yields. '
            'Bond duration measures the sensitivity of a bond price to changes in interest rates. '
            'A portfolio manager uses duration matching to hedge against rising rate environments.'
        )
    },
    {
        'id': 302,
        'title': 'Credit Risk and Portfolio Diversification',
        'body': (
            'Credit scores quantify a borrower likelihood of defaulting on a loan repayment. '
            'Lenders combine credit scores with income verification to set approval thresholds. '
            'Diversification reduces unsystematic risk by spreading exposure across uncorrelated assets. '
            'A well-diversified portfolio limits the impact of any single credit default on total returns.'
        )
    },
    {
        'id': 303,
        'title': "Bloom's Taxonomy and Formative Assessment",
        'body': (
            "Bloom's Taxonomy classifies learning objectives into six cognitive levels from remember to create. "
            'Higher-order objectives like analysis and evaluation require more demanding instructional design. '
            'Formative assessments monitor student learning continuously during instruction to inform teaching. '
            'Exit tickets aligned to specific Bloom levels give teachers actionable data within a single lesson.'
        )
    },
    {
        'id': 304,
        'title': 'Spaced Repetition and Active Learning',
        'body': (
            'Spaced repetition revisits information at increasing intervals to strengthen long-term memory. '
            'The forgetting curve shows that memory decays rapidly without scheduled review sessions. '
            'Active learning engages students through structured activities rather than passive listening. '
            'Combining spaced review with active retrieval practice produces the largest gains in retention.'
        )
    },
    {
        'id': 305,
        'title': 'EHR Systems and Medication Adherence',
        'body': (
            'Electronic Health Records store patient medical histories, diagnoses, and treatment plans digitally. '
            'EHR alerts can notify clinicians when prescribed medications have dangerous interactions. '
            'Medication adherence means patients take prescribed drugs in the correct dose at the correct time. '
            'Reminder features in EHR patient portals measurably improve adherence rates for chronic conditions.'
        )
    },
    {
        'id': 306,
        'title': 'Microservices and Containerization',
        'body': (
            'Microservices decompose a monolithic application into small, independently deployable services. '
            'Each service owns its own data store and exposes a well-defined API to other services. '
            'Containers package an application and its runtime dependencies into a portable, isolated unit. '
            'Kubernetes orchestrates containers across clusters and handles scaling and self-healing automatically.'
        )
    },
]

comparison_queries = [
    {'query': 'Why do central banks raise interest rates?', 'answer': 'control inflation', 'doc_id': 301},
    {'query': 'What measures bond price sensitivity to rate changes?', 'answer': 'duration', 'doc_id': 301},
    {"What are Bloom's Taxonomy cognitive levels?": "Bloom's Taxonomy classifies learning objectives",
     'query': "What does Bloom's Taxonomy classify?", 'answer': 'learning objectives', 'doc_id': 303},
    {'query': 'How does spaced repetition improve memory?', 'answer': 'revisits information at increasing intervals', 'doc_id': 304},
    {'query': 'What do Electronic Health Records store?', 'answer': 'medical histories, diagnoses', 'doc_id': 305},
    {'query': 'How do microservices decompose applications?', 'answer': 'independently deployable services', 'doc_id': 306},
]
# Keep only well-formed entries
comparison_queries = [q for q in comparison_queries if 'query' in q and 'answer' in q]

fixed_chunker_eval   = lambda text: fixed_size_chunking(text, chunk_size=10, overlap=0)
sentence_chunker_eval = lambda text: sentence_aware_chunking(text, max_words=25, overlap_sentences=1)
semantic_chunker_eval = lambda text: semantic_chunking(text, similarity_threshold=0.45, min_sentences=1, max_words=60)

baseline = evaluate_chunking('Fixed-size (baseline)', fixed_chunker_eval,   comparison_docs, comparison_queries)
sentence = evaluate_chunking('Sentence-aware',        sentence_chunker_eval, comparison_docs, comparison_queries)
semantic = evaluate_chunking('Semantic',              semantic_chunker_eval, comparison_docs, comparison_queries)

results = [baseline, sentence, semantic]

print('Chunking comparison — domain corpus (finance, education, healthcare, technology)')
print(f'{"Method":<22} | Top1 | Top3 |  MRR | Delta vs baseline')
for r in results:
    if r is baseline:
        delta = '---'
    else:
        delta = (f"{r['top1']-baseline['top1']:+.2f}/"
                 f"{r['top3']-baseline['top3']:+.2f}/"
                 f"{r['mrr']-baseline['mrr']:+.2f}")
    print(f"{r['method']:<22} | {r['top1']:.2f} | {r['top3']:.2f} | {r['mrr']:.2f} | {delta}")

print()
print(f'{"Method":<22} | Chunks | Avg words')
for r in results:
    print(f"{r['method']:<22} | {r['chunks']:>6} | {r['avg_len']:>9.1f}")

Relative to the fixed-size baseline, sentence-aware improves Top1/Top3/MRR by +0.33/+0.33/+0.33, while semantic improves Top1 and MRR but keeps Top3 flat. The tradeoff snapshot shows sentence-aware yields fewer, larger chunks, while semantic yields more, smaller chunks.

## Embedding model selection and retrieval measurement harness

**Embedding model selection**

| Model | Strengths | When to choose |
| --- | --- | --- |
| OpenAI `text-embedding-3-small/large` | Strong English quality, easy API integration | Default for English-first products where cost per query is acceptable |
| Cohere `embed-multilingual-v3` | Strong multilingual and domain-adaptation support | Multi-language corpora or messy real-world text |
| Open-source (BGE, E5, sentence-transformers) | No API dependency, offline use, data sovereignty | Privacy-sensitive data, high throughput, cost control |

**Retrieval patterns**

- **Top-k:** always returns exactly k results regardless of relevance score. Safe when you need a fixed-size context window for the LLM.
- **Similarity threshold:** discards results below a quality cutoff. Reduces noise but may return zero results on ambiguous queries.

**Measurement harness** — the cells below compare TF-IDF and BM25 across three metrics:
- **Top-1 accuracy:** correct answer appears in the first result.
- **Top-3 accuracy:** correct answer appears anywhere in the top 3 results.
- **MRR (Mean Reciprocal Rank):** average of 1/rank for each query; higher is better, max 1.0.

**Hallucination reduction through grounding** — when the LLM answer is assembled only from retrieved chunks, errors are anchored to the corpus. Without grounding, the model fills gaps with plausible-sounding but fabricated text. The hallucination demo below shows this difference directly.

In [46]:
threshold_chunker = lambda text: sentence_aware_chunking(text, max_words=40, overlap_sentences=1)

threshold_chunks = build_chunks(ingested_docs, threshold_chunker)
threshold_vectorizer = SimpleTfidfVectorizer()
threshold_vectors = threshold_vectorizer.fit_transform([c['text'] for c in threshold_chunks])

def retrieve_with_threshold(query, chunks, vectorizer, chunk_vectors, threshold=0.20, max_k=5):
    query_vec = vectorizer.transform([query])[0]
    scores = cosine_similarity_scores(query_vec, chunk_vectors)
    filtered = [(i, s) for i, s in enumerate(scores) if s >= threshold]
    filtered.sort(key=lambda x: x[1], reverse=True)
    filtered = filtered[:max_k]
    return [chunks[i] for i, _ in filtered]

def eval_threshold(threshold):
    hits = 0
    retrieved_counts = []
    for item in corpus_queries:
        retrieved = retrieve_with_threshold(
            item['query'],
            threshold_chunks,
            threshold_vectorizer,
            threshold_vectors,
            threshold=threshold,
            max_k=5
        )
        retrieved_counts.append(len(retrieved))
        if any(item['answer'].lower() in c['text'].lower() for c in retrieved):
            hits += 1
    hit_rate = hits / len(corpus_queries)
    avg_returned = float(np.mean(retrieved_counts))
    return hit_rate, avg_returned

def eval_top_k(k):
    hits = 0
    retrieved_counts = []
    for item in corpus_queries:
        retrieved = retrieve_top_k(
            item['query'],
            threshold_chunks,
            threshold_vectorizer,
            threshold_vectors,
            top_k=k
        )
        retrieved_counts.append(len(retrieved))
        if any(item['answer'].lower() in c['text'].lower() for c in retrieved):
            hits += 1
    hit_rate = hits / len(corpus_queries)
    avg_returned = float(np.mean(retrieved_counts))
    return hit_rate, avg_returned

topk_hit, topk_avg = eval_top_k(3)
threshold_hit, threshold_avg = eval_threshold(0.20)

print('Retrieval pattern comparison')
print('Pattern | Hit rate | Avg returned')
print(f'Top-k (k=3) | {topk_hit:.2f} | {topk_avg:.2f}')
print(f'Threshold (0.20) | {threshold_hit:.2f} | {threshold_avg:.2f}')

Retrieval pattern comparison
Pattern | Hit rate | Avg returned
Top-k (k=3) | 1.00 | 3.00
Threshold (0.20) | 1.00 | 2.00


In [ ]:

# ── BM25 (Okapi BM25) implementation ─────────────────────────────────────────
# BM25 is the industry-standard retrieval function used by Elasticsearch and
# Solr. It improves on TF-IDF by (a) capping term frequency saturation so a
# term appearing 100 times is not 100× more important than one appearing 5
# times, and (b) normalising for document length so long documents are not
# unfairly favoured.

class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1       # term frequency saturation parameter
        self.b  = b        # document length normalisation parameter
        self._corpus = []
        self._df     = {}
        self._idf    = {}
        self._avgdl  = 0
        self._N      = 0

    def fit(self, texts):
        self._corpus = [tokenize(t) for t in texts]
        self._N      = len(self._corpus)
        self._avgdl  = sum(len(d) for d in self._corpus) / max(self._N, 1)
        df = {}
        for doc in self._corpus:
            for term in set(doc):
                df[term] = df.get(term, 0) + 1
        for term, freq in df.items():
            self._df[term] = freq
            self._idf[term] = math.log(
                (self._N - freq + 0.5) / (freq + 0.5) + 1.0
            )
        return self

    def _score(self, query_terms, doc_idx):
        doc  = self._corpus[doc_idx]
        dl   = len(doc)
        tf_map = {}
        for t in doc:
            tf_map[t] = tf_map.get(t, 0) + 1
        score = 0.0
        for term in query_terms:
            if term not in self._idf:
                continue
            tf   = tf_map.get(term, 0)
            num  = tf * (self.k1 + 1)
            den  = tf + self.k1 * (1 - self.b + self.b * dl / self._avgdl)
            score += self._idf[term] * (num / den)
        return score

    def retrieve(self, query, top_k=5):
        qt     = tokenize(query)
        scores = [self._score(qt, i) for i in range(self._N)]
        ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        return ranked[:top_k]


# ── Retrieval measurement harness: TF-IDF vs BM25 ────────────────────────────
def eval_retrieval_method(method_name, retrieve_fn, queries, top_k=3):
    top1, top3, mrr_scores = 0, 0, []
    for item in queries:
        results = retrieve_fn(item['query'], top_k)
        answer  = item['answer'].lower()
        rank    = None
        for pos, chunk_or_doc in enumerate(results):
            text = (chunk_or_doc.get('text', '') or chunk_or_doc.get('body', '')).lower()
            if answer in text:
                rank = pos + 1
                break
        if rank == 1:           top1 += 1
        if rank and rank <= 3:  top3 += 1
        mrr_scores.append(1.0 / rank if rank else 0.0)
    total = len(queries)
    return {
        'method': method_name,
        'top1':   top1 / total if total else 0.0,
        'top3':   top3 / total if total else 0.0,
        'mrr':    float(np.mean(mrr_scores)) if mrr_scores else 0.0,
    }

# Build a sentence-aware index for TF-IDF baseline
harness_chunker   = lambda text: sentence_aware_chunking(text, max_words=50, overlap_sentences=1)
harness_chunks    = build_chunks(ingested_docs, harness_chunker)
tfidf_vec         = SimpleTfidfVectorizer()
tfidf_vecs        = tfidf_vec.fit_transform([c['text'] for c in harness_chunks])

def tfidf_retrieve(query, top_k=3):
    return retrieve_top_k(query, harness_chunks, tfidf_vec, tfidf_vecs, top_k=top_k)

# Build a BM25 index on the same chunks
bm25 = BM25(k1=1.5, b=0.75).fit([c['text'] for c in harness_chunks])

def bm25_retrieve(query, top_k=3):
    indices = bm25.retrieve(query, top_k=top_k)
    return [harness_chunks[i] for i in indices]

tfidf_result = eval_retrieval_method('TF-IDF',  tfidf_retrieve, corpus_queries)
bm25_result  = eval_retrieval_method('BM25',    bm25_retrieve,  corpus_queries)

print('Retrieval model comparison — 16-query multi-domain eval set')
print(f'{"Method":<10} | Top1 | Top3 |  MRR | Delta vs TF-IDF')
for r in [tfidf_result, bm25_result]:
    if r is tfidf_result:
        delta = '---'
    else:
        delta = (f"{r['top1']-tfidf_result['top1']:+.2f}/"
                 f"{r['top3']-tfidf_result['top3']:+.2f}/"
                 f"{r['mrr']-tfidf_result['mrr']:+.2f}")
    print(f"{r['method']:<10} | {r['top1']:.2f} | {r['top3']:.2f} | {r['mrr']:.2f} | {delta}")

print()
print('Interpretation:')
print('  TF-IDF weights terms by corpus frequency but treats all term occurrences equally.')
print('  BM25 saturates term frequency and normalises by doc length, which tends to')
print('  improve precision for short factual queries like the ones in this eval set.')
print('  In practice, production RAG systems use embedding models (dense retrieval)')
print('  which further outperform sparse methods, especially for paraphrased queries.')

In [47]:
thresholds = [0.05, 0.10, 0.20, 0.30, 0.40]

print('Threshold sweep')
print('Threshold | Hit rate | Avg returned')
for t in thresholds:
    hit_rate, avg_returned = eval_threshold(t)
    print(f'{t:>8.2f} | {hit_rate:.2f} | {avg_returned:.2f}')

Threshold sweep
Threshold | Hit rate | Avg returned
    0.05 | 1.00 | 4.42
    0.10 | 1.00 | 3.50
    0.20 | 1.00 | 2.00
    0.30 | 1.00 | 1.00
    0.40 | 1.00 | 1.00


As the threshold rises, fewer chunks are returned while the hit rate stays at 1.00 here. This illustrates the precision-recall tradeoff: higher thresholds reduce noise but can risk missing answers on tougher corpora.
The threshold filter keeps the hit rate at 1.00 while returning fewer chunks on average (2 vs 3), showing how similarity thresholds can cut noise without sacrificing coverage at a well-chosen cutoff.

## Query rewriting

There is no single best technique. Each method supports a different precision vs recall goal, and real systems often combine them. In the lab below, each method is compared against its own baseline (no rewrite) to show improvement rather than to rank methods.

1. Query segmentation
- What it does: breaks a query into semantic phrases.
- Best used for: improving precision by treating key phrases as a unit.

2. Query scoping
- What it does: restricts matching to specific fields (title, tags, or metadata).
- Best used for: aggressive precision to avoid irrelevant matches.

3. Query relaxation
- What it does: removes optional terms when results are too narrow.
- Best used for: increasing recall so users still get results.

4. Query expansion
- What it does: adds synonyms or related terms to reduce vocabulary mismatch.
- Best used for: improving recall when the corpus uses different wording.

Recommended pipeline:
- Segment first so the system understands core phrases.
- Scope second to enforce where those phrases must match.
- Expand or relax last if results are too narrow.

In [ ]:
# Ordered to surface baseline errors for segmentation/scoping.
rewrite_docs = {
    1: {'title': 'Intellectual Property Attorney', 'body': 'An attorney specializing in intellectual property law.'},
    2: {'title': 'IP Networking Basics', 'body': 'Internet protocol and computer networking concepts.'},
    3: {'title': 'Cute Fluffy Kittens', 'body': 'Photos of fluffy kittens and cute cats.'},
    4: {'title': 'Fluffy Kittens Playing', 'body': 'Small fluffy kittens playing together.'},
    6: {'title': 'White Shirt Dress', 'body': 'Elegant white shirt dress for women.'},
    5: {'title': 'White Dress Shirt', 'body': 'Formal white dress shirt for office wear.'},
    9: {'title': 'Artificial Intelligence', 'body': 'Machine learning is widely used in AI.'},
    7: {'title': 'Machine Learning', 'body': 'Introduction to machine learning algorithms.'},
    8: {'title': 'Learning Techniques', 'body': 'Different educational learning methods.'},
    10: {'title': 'Dress Collection', 'body': 'Women dress collection and fashion.'},
    11: {'title': 'Cute Puppies', 'body': 'Cute dogs and puppies.'}
}

expansion_queries = [
    {'query': 'ip lawer', 'doc_id': 1},
    {'query': 'ml algorithms', 'doc_id': 7}
]
relaxation_queries = [
    {'query': 'cute fluffy kittens', 'doc_id': 3},
    {'query': 'tiny fluffy kittens playing', 'doc_id': 4}
]
segmentation_queries = [
    {'query': 'white dress shirt', 'doc_id': 5}
]
scoping_queries = [
    {'query': 'machine learning', 'doc_id': 7},
    {'query': 'learning techniques', 'doc_id': 8}
]

SYNONYMS = {
    'ip': ['intellectual property'],
    'lawer': ['lawyer', 'attorney'],
    'ml': ['machine learning']
}
RELAX_TERMS = {'tiny'}
PHRASES = ['dress shirt', 'intellectual property', 'machine learning']

def lexical_search(terms, docs, phrase=None, strict_and=False, scope='all'):
    scores = {}
    for doc_id, content in docs.items():
        title = content['title'].lower()
        body = content['body'].lower()
        if scope == 'title':
            searchable = title
        elif scope == 'body':
            searchable = body
        else:
            searchable = title + ' ' + body
        matched = 0
        for term in terms:
            if term in searchable:
                matched += 1
                scores[doc_id] = scores.get(doc_id, 0) + 1
        if strict_and and matched < len(terms):
            scores.pop(doc_id, None)
            continue
        if phrase and phrase in searchable:
            scores[doc_id] = scores.get(doc_id, 0) + 5
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in ranked]

def run_retrieval(query_text, opts=None):
    if opts is None:
        terms = tokenize(query_text)
        phrase = None
        strict_and = False
        scope = 'all'
    else:
        terms = opts.get('terms', tokenize(query_text))
        phrase = opts.get('phrase')
        strict_and = opts.get('strict_and', False)
        scope = opts.get('scope', 'all')
    return lexical_search(terms, rewrite_docs, phrase=phrase, strict_and=strict_and, scope=scope)

def rewrite_expansion(query_text):
    terms = tokenize(query_text)
    expanded = []
    for term in terms:
        expanded.append(term)
        for syn in SYNONYMS.get(term, []):
            expanded.extend(tokenize(syn))
    return {'terms': expanded, 'strict_and': False}

def rewrite_relaxation(query_text):
    terms = [t for t in tokenize(query_text) if t not in RELAX_TERMS]
    return {'terms': terms, 'strict_and': False}

def rewrite_segmentation(query_text):
    phrase = None
    lowered = query_text.lower()
    for p in PHRASES:
        if p in lowered:
            phrase = p
            break
    return {'terms': tokenize(query_text), 'phrase': phrase, 'strict_and': False}

def rewrite_scoping(query_text):
    return {'terms': tokenize(query_text), 'scope': 'title', 'strict_and': False}

def evaluate_lexical(queries, opts_fn=None):
    top1 = 0
    top3 = 0
    for item in queries:
        opts = opts_fn(item['query']) if opts_fn else None
        results = run_retrieval(item['query'], opts)
        if results and results[0] == item['doc_id']:
            top1 += 1
        if item['doc_id'] in results[:3]:
            top3 += 1
    total = len(queries)
    return {
        'top1': top1 / total if total else 0.0,
        'top3': top3 / total if total else 0.0
    }

In [ ]:
groups = [
    ('Expansion', expansion_queries, lambda q: {'terms': tokenize(q), 'strict_and': True}, rewrite_expansion),
    ('Relaxation', relaxation_queries, lambda q: {'terms': tokenize(q), 'strict_and': True}, rewrite_relaxation),
    ('Segmentation', segmentation_queries, lambda q: {'terms': tokenize(q), 'strict_and': False}, rewrite_segmentation),
    ('Scoping', scoping_queries, lambda q: {'terms': tokenize(q), 'strict_and': False}, rewrite_scoping)
]

print('Query rewriting evaluation')
print('Method | Top1 before | Top1 after | Top3 before | Top3 after')
for label, queries, baseline_fn, rewrite_fn in groups:
    before = evaluate_lexical(queries, baseline_fn)
    after = evaluate_lexical(queries, rewrite_fn)
    print(f"{label:<11} | {before['top1']:.2f} | {after['top1']:.2f} | {before['top3']:.2f} | {after['top3']:.2f}")

Query rewriting evaluation
Method | Top1 before | Top1 after | Top3 before | Top3 after
Expansion   | 0.00 | 1.00 | 0.00 | 1.00
Relaxation  | 0.50 | 1.00 | 0.50 | 1.00
Segmentation | 0.00 | 1.00 | 1.00 | 1.00
Scoping     | 0.50 | 1.00 | 1.00 | 1.00


Each method improves over its own baseline, not over the other methods. Expansion and relaxation primarily boost recall, while segmentation and scoping tighten precision when queries are ambiguous.

## Lab 2: RAG pipeline build

Build a full pipeline: ingest -> chunk -> embed -> retrieve -> grounded generate. Then compare chunk size variants on retrieval quality.

## Hallucination reduction through grounding

Without retrieval, a generative system must rely entirely on parametric memory. For facts that were not seen during training — or that have changed since — it fills the gap with confident-sounding but wrong text. This is called hallucination.

RAG prevents this by **grounding**: the answer must be assembled from retrieved chunks. If the evidence is absent or contradicts the model, the model either acknowledges the gap or stays silent rather than fabricating an answer.

The demo below simulates both modes:
- **Ungrounded** — the system picks the most query-relevant single word from the index, mimicking a model with only weak memory signals and no context.
- **Grounded** — the system builds the answer sentence-by-sentence strictly from the top retrieved chunks.

In a real LLM-backed system the gap is even larger: ungrounded answers look fluent and confident, which makes hallucinations harder to detect.

In [ ]:

def ungrounded_answer(query):
    """
    Simulates a model answering from parametric memory only:
    picks the single highest-scoring term from the full corpus without
    returning any supporting text. Produces incomplete, decontextualised answers.
    """
    query_terms = set(tokenize(query))
    best_term   = None
    best_score  = 0.0
    for doc in ingested_docs:
        for term in tokenize(doc['body']):
            if term in query_terms:
                score = len(term)   # favour longer matching terms
                if score > best_score:
                    best_score = score
                    best_term  = term
    return best_term or '[no answer found]'


def grounded_answer(query, retrieve_fn, top_k=3):
    """
    Assembles an answer only from sentences inside retrieved chunks.
    If no retrieved chunk contains query terms, returns a refusal rather
    than fabricating a response.
    """
    retrieved = retrieve_fn(query, top_k)
    query_terms = set(tokenize(query))
    candidates = []
    for chunk in retrieved:
        for sentence in split_sentences(chunk['text']):
            score = sum(1 for t in tokenize(sentence) if t in query_terms)
            if score > 0:
                candidates.append((score, sentence, chunk['title']))
    if not candidates:
        return '[No supporting evidence found in the corpus — cannot answer.]'
    candidates.sort(key=lambda x: x[0], reverse=True)
    top = candidates[:2]
    answer_text = ' '.join(s for _, s, _ in top)
    source_docs = list(dict.fromkeys(title for _, _, title in top))
    return f'{answer_text}  [sources: {", ".join(source_docs)}]'


# ── Hallucination demo on cross-domain questions ──────────────────────────────
hallucination_queries = [
    {
        'domain':   'Finance',
        'question': 'Why do central banks raise interest rates?',
        'expected': 'control inflation',
    },
    {
        'domain':   'Education',
        'question': 'What does the Zone of Proximal Development describe?',
        'expected': 'gap between what a learner can do alone and with guidance',
    },
    {
        'domain':   'Healthcare',
        'question': 'What is the gold standard for evaluating medical treatments?',
        'expected': 'randomized controlled trials',
    },
    {
        'domain':   'Technology',
        'question': 'What do containers package for portability?',
        'expected': 'application and its dependencies',
    },
    {
        'domain':   'Legal',
        'question': 'What rights do individuals have under GDPR?',
        'expected': 'access, correct, and delete their data',
    },
]

print('Hallucination reduction demo')
print('=' * 90)
for item in hallucination_queries:
    ug = ungrounded_answer(item['question'])
    gr = grounded_answer(item['question'], bm25_retrieve, top_k=3)
    correct_tag = '✓' if item['expected'].split()[0].lower() in gr.lower() else '✗'
    print(f"Domain   : {item['domain']}")
    print(f"Query    : {item['question']}")
    print(f"Expected : {item['expected']}")
    print(f"Ungrounded (no context) → \"{ug}\"  ← incomplete / potentially hallucinated")
    print(f"Grounded  (RAG)         → {gr}  {correct_tag}")
    print('-' * 90)

print()
print('Key observation: the grounded answer cites the source document and quotes')
print('verbatim evidence. The ungrounded answer has no supporting context and')
print('would be trivially wrong for out-of-distribution or recently-changed facts.')

In [ ]:
def build_rag_index(docs, chunk_fn):
    chunks = build_chunks(docs, chunk_fn)
    vectorizer = SimpleTfidfVectorizer()
    vectors = vectorizer.fit_transform([c['text'] for c in chunks])
    return chunks, vectorizer, vectors

def generate_answer(query, retrieved_chunks):
    query_terms = set(tokenize(query))
    candidates = []
    for chunk in retrieved_chunks:
        for sentence in split_sentences(chunk['text']):
            score = sum(1 for t in tokenize(sentence) if t in query_terms)
            if score > 0:
                candidates.append((score, sentence))
    if not candidates:
        return retrieved_chunks[0]['text'] if retrieved_chunks else ''
    candidates.sort(key=lambda x: x[0], reverse=True)
    top_sentences = [s for _, s in candidates[:2]]
    return ' '.join(top_sentences)

In [ ]:
small_chunker = lambda text: fixed_size_chunking(text, chunk_size=8, overlap=0)
large_chunker = lambda text: fixed_size_chunking(text, chunk_size=40, overlap=5)

small = evaluate_chunking('Fixed-8', small_chunker, ingested_docs, corpus_queries)
large = evaluate_chunking('Fixed-40', large_chunker, ingested_docs, corpus_queries)

print('Chunk size comparison (same method, different sizes)')
print('Method | Chunks | Avg words | Top1 | Top3 | MRR')
for r in [small, large]:
    print(f"{r['method']:<8} | {r['chunks']:>6} | {r['avg_len']:>9.1f} | {r['top1']:.2f} | {r['top3']:.2f} | {r['mrr']:.2f}")

Chunk size comparison (same method, different sizes)
Method | Chunks | Avg words | Top1 | Top3 | MRR
Fixed-8  |    118 |       6.8 | 0.58 | 0.83 | 0.71
Fixed-40 |     60 |      13.4 | 1.00 | 1.00 | 1.00


Using the smaller fixed-size chunks as the baseline, the larger chunk size improves Top1/Top3/MRR and restores context. This reinforces that chunk size tuning is a baseline-driven improvement, not a universal winner.

In [ ]:

# ── Full RAG pipeline: end-to-end demo on domain-specific queries ─────────────
# Build index with sentence-aware chunking (best all-round in Lab 1)
rag_chunker = lambda text: sentence_aware_chunking(text, max_words=50, overlap_sentences=1)
rag_chunks, rag_vectorizer, rag_vectors = build_rag_index(ingested_docs, rag_chunker)

domain_demo_queries = [
    {'domain': 'Finance',    'query': 'Why do central banks raise interest rates?'},
    {'domain': 'Finance',    'query': 'What measures a bond sensitivity to rate changes?'},
    {'domain': 'Education',  'query': 'What does the Zone of Proximal Development describe?'},
    {'domain': 'Education',  'query': 'How does spaced repetition strengthen memory?'},
    {'domain': 'Healthcare', 'query': 'What is the gold standard for evaluating medical treatments?'},
    {'domain': 'Technology', 'query': 'What do containers package for portability?'},
    {'domain': 'Legal',      'query': 'What rights do individuals have under GDPR?'},
]

print('Full RAG pipeline — domain-specific query demo')
print('Pipeline: ingest → sentence-aware chunk → TF-IDF embed → top-3 retrieve → grounded generate')
print('=' * 90)

for item in domain_demo_queries:
    retrieved = retrieve_top_k(item['query'], rag_chunks, rag_vectorizer, rag_vectors, top_k=3)
    answer    = generate_answer(item['query'], retrieved)
    top_doc   = retrieved[0]['title'] if retrieved else 'n/a'
    print(f"[{item['domain']}]  {item['query']}")
    print(f"  → Top doc : {top_doc}")
    print(f"  → Answer  : {answer}")
    print()

The top chunk contains the exact evidence sentence, and the generated answer cites it directly, showing grounded generation from retrieved context.